# MVP02 — Coleta e Ingestão na Camada Staging

Coleta os arquivos de origem no Google Drive via API e os materializa como tabelas Delta
na camada `staging` do Data Lakehouse.

**Posição no pipeline:** segundo notebook. Requer `MVP01-Setup` executado.

**Estratégia de carga:** *full refresh*. Cada execução limpa o Volume, rebaixa os três
arquivos e substitui integralmente as tabelas de staging. O volume de dados (pouco mais
de 4 mil registros somados) não justifica carga incremental, e o refresh completo
simplifica a reprodutibilidade.

**Princípio da camada:** a staging é uma cópia fiel da origem. Todas as colunas são
gravadas como texto, sem inferência de tipo. A tipagem é responsabilidade da camada
Silver, onde está documentada e versionada.

## 📚 Sobre este Notebook

Este notebook implementa o processo completo de **ingestão de dados** seguindo a **Arquitetura Medalhão (Medallion Architecture)**.

### 🎯 Objetivo
Baixar dados de planilhas Excel do Google Drive e carregá-los na **camada Staging** do Data Lakehouse.
As demais camadas (Bronze, Silver e Gold) serão implementadas em notebooks subsequentes.

### Arquitetura Medalhão

A arquitetura medalhão organiza os dados em camadas:

* **📂 Staging (Chegada)**: Dados brutos copiados diretamente do Google Drive (Este notebook)
  - Cópia fiel da origem, sem nenhuma transformação
  - Área temporária de transbordo
  - Inclui metadados de ingestão

* **🥉 Bronze (Raw)**: Dados brutos com metadados de rastreabilidade (Notebook MVP03-Bronze)
  - Cópia fiel da origem com controle de versão
  - Mantém histórico completo
  - Implementada em notebook próprio
  
* **🥈 Silver (Cleaned)**: Dados limpos e validados (Notebook MVP04-Silver)
  - Dados padronizados
  - Qualidade garantida
  - Implementada em notebook próprio
  
* **🥇 Gold (Curated)**: Dados agregados e otimizados (Notebook MVP05-Gold)
  - Métricas de negócio
  - Dados prontos para consumo
  - Implementada em notebook próprio

## 🔧 Etapa 1: Preparação do Ambiente

Instalar as bibliotecas Python necessárias para autenticação OAuth 2.0 com o Google, acesso à API do Google Drive e Leitura de arquivos Excel(.xlsx)

## Bibliotecas instaladas:
* `google-auth` - Autenticação com Google Cloud
* `google-auth-oauthlib` - Fluxo OAuth 2.0
* `google-auth-httplib2` - Transporte HTTP
* `google-api-python-client` - Cliente da API do Google Drive
* `openpyxl` - Leitura de arquivos Excel

`dbutils.library.restartPython()` reinicia o interpretador para que os pacotes recém-instalados
fiquem disponíveis aos imports. Por isso os imports ficam na célula seguinte, e não nesta.


In [0]:
# Instalar bibliotecas para Google Drive e Excel
%pip install google-auth google-auth-oauthlib google-auth-httplib2 google-api-python-client openpyxl --quiet

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# Reinicia o interpretador para que os pacotes instalados acima fiquem disponíveis.
# Atenção: isso limpa todas as variáveis da sessão, por isso vem antes de qualquer definição.
dbutils.library.restartPython()

In [0]:
# Imports concentrados em uma única célula, após o restartPython.

# Google Cloud / Drive API
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# Biblioteca padrão
import io
import os
import json
import glob

# Processamento
import pandas as pd
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType

print("Bibliotecas importadas.")

Bibliotecas importadas.


## 📁 Etapa 2: Preparação do Armazenamento

* Definir o Catalogo e schema que será usado. 
* Criar o volume dentro do schema selecionado. 

Os arquivos Excel baixados do Google Drive são armazenados em um **Volume do Unity Catalog** (`mvpDataEngineer.staging.leituras`), garantindo persistência, governança e disponibilidade entre sessões.

In [0]:

CATALOGO = "mvpdataengineer"
SCHEMA   = "staging"
VOLUME   = "leituras"
FOLDER_ID = ""

BASE_PATH = f"/Volumes/{CATALOGO}/{SCHEMA}/{VOLUME}"

# Mapeamento explícito arquivo de origem -> tabela de destino.
# Evita que uma planilha renomeada no Drive crie silenciosamente uma tabela nova.
MAPA_ARQUIVO_TABELA = {
    "LEITURA_PLUVIOMETROS.xlsx": "leitura_pluviometros",
    "MEDICAO_BACIAS.xlsx":       "medicao_bacias",
    "MEDICAO_LAJE.xlsx":         "medicao_laje",
}

print(f"Catálogo ......: {CATALOGO}")
print(f"Schema ........: {SCHEMA}")
print(f"Volume ........: {BASE_PATH}")
print(f"Arquivos ......: {list(MAPA_ARQUIVO_TABELA)}")

Catálogo ......: mvpdataengineer
Schema ........: staging
Volume ........: /Volumes/mvpdataengineer/staging/leituras
Arquivos ......: ['LEITURA_PLUVIOMETROS.xlsx', 'MEDICAO_BACIAS.xlsx', 'MEDICAO_LAJE.xlsx']


## 🔐 Etapa 3: Autenticação com Google Drive

Para essa etapa foi utilizado o **Service Account** que é uma conta de serviço do Google Cloud que permite autenticação automática sem intervenção do usuário.

##### Como funciona?
Cria-se um projeto no Google Cloud, habilita-se a Google Drive API, gera-se uma Service Account com chave JSON e compartilha-se a pasta do Drive com essa conta de serviço.


***⚠️ Como não é um ambiente produtivo, as credenciais foram utilizadas diretamente no código, mas por segurança poreria ser utilizado o Databricks Secrets.***

In [0]:
SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]


CREDENCIAL_JSON = {
#--------------------------------------------------
# 🔐 Por segurança as credenciais foram retiradas.
#--------------------------------------------------
}

# ---------------------------------------------------------------------------

def carregar_credencial():
    """Retorna o dicionário de credenciais, priorizando o secret scope."""
    try:
        bruto = dbutils.secrets.get(scope="gdrive", key="service_account")
        print("Credencial obtida do secret scope 'gdrive'.")
        return json.loads(bruto)
    except Exception:
        pass

    if CREDENCIAL_JSON.get("private_key"):
        return CREDENCIAL_JSON

    raise RuntimeError(
        "Credencial não encontrada."
    )


credentials = service_account.Credentials.from_service_account_info(
    carregar_credencial(), scopes=SCOPES
)

print("Credenciais configuradas.")

⚠️  Credencial obtida do bloco embutido no notebook.
    Apague o bloco e revogue a chave antes de publicar o repositório.
Credenciais configuradas.


## 🛠️ Etapa 3: Funções de Utilitário

###### 1️⃣ `list_files_in_folder()`
**Propósito:** Listar arquivos de uma pasta

**Parâmetros:**
* `service` - Conexão com Google Drive API
* `folder_id` - ID da pasta (opcional)
* `file_type` - Filtro por tipo MIME (opcional)

**Retorna:** Lista de arquivos com metadados (id, nome, tamanho, data)

###### 2️⃣ `download_file()`
**Propósito:** Baixar um arquivo específico

**Parâmetros:**
* `service` - Conexão com Google Drive API
* `file_id` - ID do arquivo
* `file_name` - Nome do arquivo
* `destination_path` - Caminho local de destino

**Retorna:** Caminho completo do arquivo baixado

In [0]:
def listar_arquivos_da_pasta(service, folder_id=None, tipo_mime=None):
    """
        Baixa um arquivo do Google Drive
    
    Args:
        service: Serviço do Google Drive API
        file_id: ID do arquivo no Google Drive
        file_name: Nome do arquivo
        destination_path: Caminho de destino local
    
    Returns:
        Caminho completo do arquivo baixado
    """
    filtros = ["mimeType != 'application/vnd.google-apps.folder'", "trashed = false"]
    if folder_id:
        filtros.append(f"'{folder_id}' in parents")
    if tipo_mime:
        filtros.append(f"mimeType = '{tipo_mime}'")

    resultado = service.files().list(
        q=" and ".join(filtros),
        pageSize=100,
        fields="files(id, name, mimeType, size, createdTime, modifiedTime)",
    ).execute()

    return resultado.get("files", [])


def baixar_arquivo(service, file_id, nome_arquivo, destino):
    """Baixa um arquivo do Google Drive para o caminho de destino.

    Returns:
        str | None: caminho completo do arquivo baixado, ou None em caso de falha.
    """
    try:
        requisicao = service.files().get_media(fileId=file_id)
        caminho = os.path.join(destino, nome_arquivo)

        with io.FileIO(caminho, "wb") as arquivo:
            downloader = MediaIoBaseDownload(arquivo, requisicao)
            concluido = False
            while not concluido:
                status, concluido = downloader.next_chunk()
                if status:
                    print(f"    {int(status.progress() * 100):3d}%  {nome_arquivo}")

        print(f"  ✓ {nome_arquivo}")
        return caminho

    except Exception as erro:
        print(f"  ✗ Falha ao baixar {nome_arquivo}: {erro}")
        return None


print("Funções utilitárias definidas.")

Funções utilitárias definidas.


## Etapa 5 — Preparação do Volume

Os arquivos são armazenados em um **Volume do Unity Catalog**, e não no sistema de
arquivos efêmero do cluster. O Volume garante persistência entre sessões, governança de
acesso e rastreabilidade — e é a área de armazenamento disponível no Databricks Free
Edition, onde o DBFS não está acessível.

O Volume é limpo antes de cada carga. Sem isso, uma falha no download deixaria os
arquivos da execução anterior no diretório, e eles seriam reingeridos como se fossem novos.

In [0]:
spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql(f"USE SCHEMA {SCHEMA}")

spark.sql(f"""
    CREATE VOLUME IF NOT EXISTS {CATALOGO}.{SCHEMA}.{VOLUME}
    COMMENT 'Arquivos Excel brutos baixados do Google Drive, antes da ingestão nas tabelas Delta da camada Staging'
""")

dbutils.fs.mkdirs(BASE_PATH)

# Limpeza dos arquivos de execuções anteriores
residuais = glob.glob(f"{BASE_PATH}/*.xlsx")
for caminho in residuais:
    os.remove(caminho)

print(f"Volume pronto: {BASE_PATH}")
print(f"Arquivos residuais removidos: {len(residuais)}")

Volume pronto: /Volumes/mvpdataengineer/staging/leituras
Arquivos residuais removidos: 0


## 🔍  Etapa 6 — Descoberta da pasta de origem

Cada pasta do Google Drive tem um identificador único (*folder ID*) usado pela API.
A célula abaixo lista as pastas visíveis à Service Account — que são apenas aquelas
explicitamente compartilhadas com ela.

Se o widget `folder_id` estiver vazio, a pasta é localizada automaticamente pelo nome.

In [0]:
service = build("drive", "v3", credentials=credentials)

resultado = service.files().list(
    q="mimeType = 'application/vnd.google-apps.folder' and trashed = false",
    pageSize=100,
    fields="files(id, name)",
).execute()

pastas = resultado.get("files", [])

print(f"Pastas acessíveis à Service Account: {len(pastas)}\n")
print(f"{'Nome':<40} | Folder ID")
print("-" * 80)
for pasta in pastas:
    print(f"{pasta['name']:<40} | {pasta['id']}")

# Resolve o folder_id quando o widget não foi preenchido
if not FOLDER_ID:
    if len(pastas) == 1:
        FOLDER_ID = pastas[0]["id"]
        print(f"\nfolder_id resolvido automaticamente: {FOLDER_ID} ({pastas[0]['name']})")
    else:
        raise RuntimeError(
            "Informe o folder_id no widget: mais de uma pasta acessível à Service Account."
        )

Pastas acessíveis à Service Account: 1

Nome                                     | Folder ID
--------------------------------------------------------------------------------
DataEngineer                             | 1iUl0QnTILpa4KkTirMq2B1V0KUp8Ueuv

folder_id resolvido automaticamente: 1iUl0QnTILpa4KkTirMq2B1V0KUp8Ueuv (DataEngineer)


## 📥 Etapa 7 — Download dos arquivos

A célula abaixo executa o **download automatizado** dos arquivos Excel da pasta `DataEngineer` no Google Drive. As principais ações são:

1. **Configura o serviço do Google Drive** usando as credenciais já autenticadas.
2. **Define a pasta de origem** pelo `folder_id` identificado na etapa anterior.
3. **Especifica os 3 arquivos-alvo**:
   - `LEITURA_PLUVIOMETROS.xlsx` — dados de pluviometria
   - `MEDICAO_BACIAS.xlsx` — dados das bacias de evaporação
   - `MEDICAO_LAJE.xlsx` — medições de cristalizadores
4. **Lista todos os arquivos** da pasta usando a função `list_files_in_folder()`.
5. **Filtra** apenas os arquivos desejados e avisa caso algum não seja encontrado.
6. **Baixa cada arquivo** com a função `download_file()`, salvando no diretório temporário (`base_path`).
7. **Registra metadados** de cada download (nome, caminho, tamanho em KB e data de modificação).
8. **Exibe um resumo final** em formato de Spark DataFrame com os arquivos baixados.

#### ✅ Validação:
Se algum arquivo não for encontrado, o script avisará quais estão faltando.

In [0]:
print("Buscando arquivos na pasta de origem...\n")

arquivos_na_pasta = listar_arquivos_da_pasta(service, folder_id=FOLDER_ID)
print(f"Arquivos na pasta: {len(arquivos_na_pasta)}")
for a in arquivos_na_pasta:
    print(f"  - {a['name']}")

esperados = set(MAPA_ARQUIVO_TABELA)
a_baixar = [a for a in arquivos_na_pasta if a["name"] in esperados]

faltando = esperados - {a["name"] for a in a_baixar}
if faltando:
    raise RuntimeError(f"Arquivos esperados não encontrados na pasta de origem: {sorted(faltando)}")

print(f"\nBaixando {len(a_baixar)} arquivo(s)...\n")

baixados = []
for arquivo in a_baixar:
    caminho = baixar_arquivo(service, arquivo["id"], arquivo["name"], BASE_PATH)
    if caminho:
        baixados.append({
            "arquivo":      arquivo["name"],
            "caminho":      caminho,
            "tamanho_kb":   round(int(arquivo.get("size", 0)) / 1024, 2),
            "modificado_em": arquivo.get("modifiedTime", "N/A"),
        })

# Guarda contra carga parcial
if len(baixados) != len(esperados):
    raise RuntimeError(
        f"Download incompleto: {len(baixados)} de {len(esperados)} arquivos. "
        "Ingestão abortada para não gerar carga parcial."
    )

print(f"\nDownload concluído: {len(baixados)} arquivo(s).")
display(spark.createDataFrame(baixados))

Buscando arquivos na pasta de origem...

Arquivos na pasta: 3
  - MEDICAO_LAJE.xlsx
  - LEITURA_PLUVIOMETROS.xlsx
  - MEDICAO_BACIAS.xlsx

Baixando 3 arquivo(s)...

    100%  MEDICAO_LAJE.xlsx
  ✓ MEDICAO_LAJE.xlsx
    100%  LEITURA_PLUVIOMETROS.xlsx
  ✓ LEITURA_PLUVIOMETROS.xlsx
    100%  MEDICAO_BACIAS.xlsx
  ✓ MEDICAO_BACIAS.xlsx

Download concluído: 3 arquivo(s).


arquivo,caminho,modificado_em,tamanho_kb
MEDICAO_LAJE.xlsx,/Volumes/mvpdataengineer/staging/leituras/MEDICAO_LAJE.xlsx,2026-09-12T00:58:12.000Z,23.12
LEITURA_PLUVIOMETROS.xlsx,/Volumes/mvpdataengineer/staging/leituras/LEITURA_PLUVIOMETROS.xlsx,2026-09-12T00:48:01.000Z,119.02
MEDICAO_BACIAS.xlsx,/Volumes/mvpdataengineer/staging/leituras/MEDICAO_BACIAS.xlsx,2026-09-12T00:14:52.000Z,36.25


## 📂 Etapa 8: Ingestão na Camada Staging

Objetivo: Carregar arquivos Excel do Google Drive para tabelas Delta na camada Staging, mantendo os dados brutos + metadados de rastreabilidade.

As transformações para Bronze, Silver e Gold serão feitas em notebooks próprios.

### 🎯 Tabelas criadas:
1. `mvpDataEngineer.staging.leitura_pluviometros`
2. `mvpDataEngineer.staging.medicao_bacias`
3. `mvpDataEngineer.staging.medicao_laje`

### 📊 Metadados adicionados:
* **data_ingestao** - Timestamp da carga
* **origem** - Fonte dos dados (google_drive)
* **arquivo_origem** - Nome do arquivo original

In [0]:
print("Iniciando ingestão na camada Staging...\n")
print("=" * 80)

resumo_ingestao = []

for nome_arquivo, nome_tabela in MAPA_ARQUIVO_TABELA.items():
    caminho = os.path.join(BASE_PATH, nome_arquivo)
    print(f"\n{nome_arquivo}")

    # 1. Leitura sem inferência de tipo
    pdf = pd.read_excel(caminho, sheet_name=0, dtype=str)

    # 2. NaN do pandas -> None, para virar NULL no Spark.
    #    O hífen literal '-' é preservado como texto.
    pdf = pdf.astype(object).where(pd.notna(pdf), None)

    print(f"  Linhas lidas .....: {len(pdf)}")
    print(f"  Colunas de origem : {len(pdf.columns)}")

    # 3. Schema Spark explícito: tudo texto, nada inferido
    schema = StructType([StructField(c, StringType(), True) for c in pdf.columns])
    sdf = spark.createDataFrame(pdf, schema=schema)

    # 4. Metadados de rastreabilidade
    sdf = (sdf
           .withColumn("data_ingestao", current_timestamp())
           .withColumn("origem", lit("google_drive"))
           .withColumn("arquivo_origem", lit(nome_arquivo)))

    # 5. Gravação full refresh
    tabela = f"{CATALOGO}.{SCHEMA}.{nome_tabela}"
    (sdf.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(tabela))

    total = sdf.count()
    print(f"  Tabela ...........: {tabela}")
    print(f"  Registros gravados: {total}")

    resumo_ingestao.append({
        "arquivo": nome_arquivo,
        "tabela": nome_tabela,
        "registros": total,
        "colunas": len(sdf.columns),
    })

print("\n" + "=" * 80)
print("Ingestão concluída.")
display(spark.createDataFrame(resumo_ingestao))

Iniciando ingestão na camada Staging...


LEITURA_PLUVIOMETROS.xlsx
  Linhas lidas .....: 3445
  Colunas de origem : 8
  Tabela ...........: mvpdataengineer.staging.leitura_pluviometros
  Registros gravados: 3445

MEDICAO_BACIAS.xlsx
  Linhas lidas .....: 506
  Colunas de origem : 12
  Tabela ...........: mvpdataengineer.staging.medicao_bacias
  Registros gravados: 506

MEDICAO_LAJE.xlsx
  Linhas lidas .....: 226
  Colunas de origem : 16
  Tabela ...........: mvpdataengineer.staging.medicao_laje
  Registros gravados: 226

Ingestão concluída.


arquivo,colunas,registros,tabela
LEITURA_PLUVIOMETROS.xlsx,11,3445,leitura_pluviometros
MEDICAO_BACIAS.xlsx,15,506,medicao_bacias
MEDICAO_LAJE.xlsx,19,226,medicao_laje


## Etapa 9 — Documentação das tabelas no Unity Catalog

Comentários registrados no catálogo tornam as tabelas autodescritivas: eles aparecem no
Catalog Explorer, no autocomplete do editor SQL e nas APIs de metadados.

As descrições nesta camada registram o significado do campo **na origem**, sem antecipar
o tratamento que a Silver aplicará.

In [0]:
%sql
COMMENT ON TABLE mvpdataengineer.staging.leitura_pluviometros IS
  'Camada de chegada. Leituras pluviométricas das estações da salina, exportadas do formulário eletrônico de controle produtivo. Todas as colunas como texto, sem transformação.';

COMMENT ON COLUMN mvpdataengineer.staging.leitura_pluviometros.PROTOCOLO IS 'Identificador da solicitação no formulário de origem. Uma solicitação agrupa várias leituras.';
COMMENT ON COLUMN mvpdataengineer.staging.leitura_pluviometros.SOLICDATA IS 'Data e hora do lançamento no sistema, no formato dd/MM/yyyy HH:mm:ss. Distinta da data da leitura.';
COMMENT ON COLUMN mvpdataengineer.staging.leitura_pluviometros.STATUS IS 'Situação no fluxo de aprovação do formulário.';
COMMENT ON COLUMN mvpdataengineer.staging.leitura_pluviometros.UNIDADE IS 'Unidade da empresa à qual a estação pertence.';
COMMENT ON COLUMN mvpdataengineer.staging.leitura_pluviometros.ESTACAO IS 'Rótulo da estação pluviométrica conforme registrado no formulário.';
COMMENT ON COLUMN mvpdataengineer.staging.leitura_pluviometros.DATAREGISTRO IS 'Data da leitura em campo, no formato dd/MM/yyyy.';
COMMENT ON COLUMN mvpdataengineer.staging.leitura_pluviometros.MILIMETROS IS 'Precipitação medida, em milímetros. Decimal com vírgula. Ausência representada por hífen.';
COMMENT ON COLUMN mvpdataengineer.staging.leitura_pluviometros.OBSERVACOES IS 'Campo livre do formulário.';

COMMENT ON COLUMN mvpdataengineer.staging.leitura_pluviometros.data_ingestao IS 'Timestamp da carga na camada Staging.';
COMMENT ON COLUMN mvpdataengineer.staging.leitura_pluviometros.origem IS 'Sistema de origem dos dados.';
COMMENT ON COLUMN mvpdataengineer.staging.leitura_pluviometros.arquivo_origem IS 'Nome do arquivo Excel que deu origem à linha.';

In [0]:
%sql
COMMENT ON TABLE mvpdataengineer.staging.medicao_bacias IS
  'Camada de chegada. Medição diária das bacias de evaporação pelo método de reposição até marca de referência, exportada do formulário eletrônico de controle produtivo. Todas as colunas como texto, sem transformação.';

COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.PROTOCOLO IS 'Identificador da solicitação no formulário de origem. Uma solicitação agrupa vários dias e os dois pontos de medição.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.SOLICDATA IS 'Data e hora do lançamento no sistema, no formato dd/MM/yyyy HH:mm:ss. Distinta da data da medição.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.STATUS IS 'Situação no fluxo de aprovação do formulário.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.UNIDADE IS 'Unidade da empresa à qual a bacia pertence.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.BACIA IS 'Ponto de medição de evaporação.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.DATAREGISTRO IS 'Data da medição em campo, no formato dd/MM/yyyy.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.AGUADOCE_COLOCADO IS 'Volume de água doce reposto até a marca de referência, em litros. Corresponde à evaporação do dia. Ausência representada por hífen.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.AGUADOCE_RETIRADO IS 'Volume de água doce retirado por excedente de chuva, em litros. Mutuamente exclusivo com AGUADOCE_COLOCADO. Ausência representada por hífen.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.SALMOURA_COLOCADO IS 'Volume de salmoura reposto até a marca de referência, em litros. Corresponde à evaporação do dia. Ausência representada por hífen.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.SALMOURA_RETIRADO IS 'Volume de salmoura retirado por excedente de chuva, em litros. Mutuamente exclusivo com SALMOURA_COLOCADO. Ausência representada por hífen.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.SALMOURA_DENSIDADE IS 'Concentração da salmoura em graus Baumé (°Bé), medida com densímetro. Decimal com vírgula. Ausência representada por hífen.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.OBSERVACOES IS 'Campo livre do formulário.';

COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.data_ingestao IS 'Timestamp da carga na camada Staging.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.origem IS 'Sistema de origem dos dados.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_bacias.arquivo_origem IS 'Nome do arquivo Excel que deu origem à linha.';

In [0]:
%sql
COMMENT ON TABLE mvpdataengineer.staging.medicao_laje IS
  'Camada de chegada. Medição mensal da laje de sal dos cristalizadores por meio de 8 réguas, exportada do formulário eletrônico de controle produtivo. Todas as colunas como texto, sem transformação.';

COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.PROTOCOLO IS 'Identificador da solicitação no formulário de origem. Uma solicitação agrupa os 25 cristalizadores de uma campanha de medição.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.SOLICDATA IS 'Data e hora do lançamento no sistema, no formato dd/MM/yyyy HH:mm:ss. Distinta da data da medição.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.STATUS IS 'Situação no fluxo de aprovação do formulário.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.UNIDADE IS 'Unidade da empresa à qual o cristalizador pertence.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.DATAMEDICAO IS 'Data da medição em campo, no formato dd/MM/yyyy.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.CRISTALIZADOR IS 'Identificador do cristalizador medido.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.AREAHECTARES IS 'Área do cristalizador em hectares. Constante por cristalizador.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.REGUA_1 IS 'Altura livre da régua 1 acima da laje, em centímetros. Régua de 1 metro, portanto 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.REGUA_2 IS 'Altura livre da régua 2 acima da laje, em centímetros. Régua de 1 metro, portanto 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.REGUA_3 IS 'Altura livre da régua 3 acima da laje, em centímetros. Régua de 1 metro, portanto 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.REGUA_4 IS 'Altura livre da régua 4 acima da laje, em centímetros. Régua de 1 metro, portanto 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.REGUA_5 IS 'Altura livre da régua 5 acima da laje, em centímetros. Régua de 1 metro, portanto 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.REGUA_6 IS 'Altura livre da régua 6 acima da laje, em centímetros. Régua de 1 metro, portanto 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.REGUA_7 IS 'Altura livre da régua 7 acima da laje, em centímetros. Régua de 1 metro, portanto 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.REGUA_8 IS 'Altura livre da régua 8 acima da laje, em centímetros. Régua de 1 metro, portanto 100 é o limite superior da leitura.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.OBSERVACOES IS 'Campo livre do formulário. Registra eventualmente os dias de fabricação e a ocorrência de colheita.';

COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.data_ingestao IS 'Timestamp da carga na camada Staging.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.origem IS 'Sistema de origem dos dados.';
COMMENT ON COLUMN mvpdataengineer.staging.medicao_laje.arquivo_origem IS 'Nome do arquivo Excel que deu origem à linha.';

## Etapa 10 — Validação da carga

Confere o volume ingerido, a origem e o momento da carga de cada tabela.

In [0]:
%sql
SELECT 'leitura_pluviometros' AS tabela, COUNT(*) AS registros,
       MIN(arquivo_origem) AS arquivo, MAX(data_ingestao) AS ingerido_em
FROM mvpdataengineer.staging.leitura_pluviometros
UNION ALL
SELECT 'medicao_bacias', COUNT(*), MIN(arquivo_origem), MAX(data_ingestao)
FROM mvpdataengineer.staging.medicao_bacias
UNION ALL
SELECT 'medicao_laje', COUNT(*), MIN(arquivo_origem), MAX(data_ingestao)
FROM mvpdataengineer.staging.medicao_laje
ORDER BY tabela;

tabela,registros,arquivo,ingerido_em
leitura_pluviometros,3445,LEITURA_PLUVIOMETROS.xlsx,2026-09-26T21:40:03.475Z
medicao_bacias,506,MEDICAO_BACIAS.xlsx,2026-09-26T21:40:18.426Z
medicao_laje,226,MEDICAO_LAJE.xlsx,2026-09-26T21:40:23.594Z


Confirmação de que nenhuma coluna foi tipada na chegada: todas devem aparecer como `STRING`,
exceto os três metadados acrescentados pelo pipeline.

In [0]:
%sql
SELECT table_name, column_name, data_type
FROM mvpdataengineer.information_schema.columns
WHERE table_schema = 'staging'
ORDER BY table_name, ordinal_position;

table_name,column_name,data_type
leitura_pluviometros,PROTOCOLO,STRING
leitura_pluviometros,SOLICDATA,STRING
leitura_pluviometros,STATUS,STRING
leitura_pluviometros,UNIDADE,STRING
leitura_pluviometros,ESTACAO,STRING
leitura_pluviometros,DATAREGISTRO,STRING
leitura_pluviometros,MILIMETROS,STRING
leitura_pluviometros,OBSERVACOES,STRING
leitura_pluviometros,data_ingestao,TIMESTAMP
leitura_pluviometros,origem,STRING


In [0]:
%sql
SELECT * FROM mvpdataengineer.staging.medicao_bacias LIMIT 10;

PROTOCOLO,SOLICDATA,STATUS,UNIDADE,BACIA,DATAREGISTRO,AGUADOCE_COLOCADO,AGUADOCE_RETIRADO,SALMOURA_COLOCADO,SALMOURA_RETIRADO,SALMOURA_DENSIDADE,OBSERVACOES,data_ingestao,origem,arquivo_origem
280396,02/01/2026 13:08:09,Solicitação Finalizada,RN,BACIA1,01/01/2026,9,-,5,-,"25,2",null,2026-09-26T21:40:18.426Z,google_drive,MEDICAO_BACIAS.xlsx
280396,02/01/2026 13:08:09,Solicitação Finalizada,RN,BACIA1,02/01/2026,11,-,6,-,"25,4",null,2026-09-26T21:40:18.426Z,google_drive,MEDICAO_BACIAS.xlsx
280901,05/01/2026 16:44:33,Solicitação Finalizada,RN,BACIA1,03/01/2026,14,-,6,-,"25,6",null,2026-09-26T21:40:18.426Z,google_drive,MEDICAO_BACIAS.xlsx
280901,05/01/2026 16:44:33,Solicitação Finalizada,RN,BACIA1,04/01/2026,10,-,5,-,"26,0",null,2026-09-26T21:40:18.426Z,google_drive,MEDICAO_BACIAS.xlsx
280901,05/01/2026 16:44:33,Solicitação Finalizada,RN,BACIA1,05/01/2026,11,-,5,-,"26,2",null,2026-09-26T21:40:18.426Z,google_drive,MEDICAO_BACIAS.xlsx
281043,06/01/2026 15:00:08,Validar Dados,RN,BACIA1,06/01/2026,13,null,7,null,"26,4",null,2026-09-26T21:40:18.426Z,google_drive,MEDICAO_BACIAS.xlsx
281456,08/01/2026 13:09:54,Solicitação Finalizada,RN,BACIA1,07/01/2026,15,-,6,-,"26,6",null,2026-09-26T21:40:18.426Z,google_drive,MEDICAO_BACIAS.xlsx
281456,08/01/2026 13:09:54,Solicitação Finalizada,RN,BACIA1,08/01/2026,13,-,7,-,"26,8",null,2026-09-26T21:40:18.426Z,google_drive,MEDICAO_BACIAS.xlsx
281768,09/01/2026 15:54:37,Solicitação Finalizada,RN,BACIA1,09/01/2026,15,-,6,-,"27,2",null,2026-09-26T21:40:18.426Z,google_drive,MEDICAO_BACIAS.xlsx
281940,10/01/2026 15:26:15,Solicitação Finalizada,RN,BACIA1,10/01/2026,11,-,4,-,"27,4",null,2026-09-26T21:40:18.426Z,google_drive,MEDICAO_BACIAS.xlsx


In [0]:
%sql
SELECT * FROM mvpdataengineer.staging.medicao_laje LIMIT 10;

PROTOCOLO,SOLICDATA,STATUS,UNIDADE,DATAMEDICAO,CRISTALIZADOR,AREAHECTARES,REGUA_1,REGUA_2,REGUA_3,REGUA_4,REGUA_5,REGUA_6,REGUA_7,REGUA_8,OBSERVACOES,data_ingestao,origem,arquivo_origem
292552,03/03/2026 11:24:53,Solicitação Finalizada,RN,31/01/2026,01,16.15,88,88,88,88,88,88,88,88,null,2026-09-26T21:40:23.594Z,google_drive,MEDICAO_LAJE.xlsx
292543,03/03/2026 11:18:30,Solicitação Finalizada,RN,28/02/2026,01,16.15,86,86,86,85,85,86,86,86,null,2026-09-26T21:40:23.594Z,google_drive,MEDICAO_LAJE.xlsx
300223,09/04/2026 11:23:14,Solicitação Finalizada,RN,31/03/2026,01,16.15,83,83,83,82,83,84,84,83,null,2026-09-26T21:40:23.594Z,google_drive,MEDICAO_LAJE.xlsx
304213,27/04/2026 11:20:29,Solicitação Finalizada,RN,31/03/2026,01,16.15,83,83,83,82,83,84,84,83,null,2026-09-26T21:40:23.594Z,google_drive,MEDICAO_LAJE.xlsx
306320,07/05/2026 09:35:58,Solicitação Finalizada,RN,30/04/2026,01,16.15,82,82,82,82,82,82,82,82,null,2026-09-26T21:40:23.594Z,google_drive,MEDICAO_LAJE.xlsx
323656,02/06/2026 10:25:29,Solicitação Finalizada,RN,25/05/2026,01,16.15,83,83,84,84,84,83,83,83,null,2026-09-26T21:40:23.594Z,google_drive,MEDICAO_LAJE.xlsx
330749,02/07/2026 10:24:03,Solicitação Finalizada,RN,26/06/2026,01,16.15,80,81,81,81,81,81,81,81,null,2026-09-26T21:40:23.594Z,google_drive,MEDICAO_LAJE.xlsx
338252,03/08/2026 09:38:41,Solicitação Finalizada,RN,27/07/2026,01,16.15,99,98,98,99,99,98,98,99,null,2026-09-26T21:40:23.594Z,google_drive,MEDICAO_LAJE.xlsx
345518,02/09/2026 10:25:40,Solicitação Finalizada,RN,31/08/2026,01,16.15,95,95,95,95,96,96,96,96,null,2026-09-26T21:40:23.594Z,google_drive,MEDICAO_LAJE.xlsx
292552,03/03/2026 11:24:53,Solicitação Finalizada,RN,31/01/2026,02,16.18,83,83,83,83,84,84,84,84,null,2026-09-26T21:40:23.594Z,google_drive,MEDICAO_LAJE.xlsx


In [0]:
%sql
SELECT * FROM mvpdataengineer.staging.leitura_pluviometros LIMIT 10;

PROTOCOLO,SOLICDATA,STATUS,UNIDADE,ESTACAO,DATAREGISTRO,MILIMETROS,OBSERVACOES,data_ingestao,origem,arquivo_origem
190825,23/05/2025 10:53:42,Solicitação Finalizada,RN,PLUVIOMETRO1,06/01/2014,"4,0",null,2026-09-26T21:40:03.475Z,google_drive,LEITURA_PLUVIOMETROS.xlsx
190825,23/05/2025 10:53:42,Solicitação Finalizada,RN,PLUVIOMETRO2,06/01/2014,"2,0",null,2026-09-26T21:40:03.475Z,google_drive,LEITURA_PLUVIOMETROS.xlsx
190825,23/05/2025 10:53:42,Solicitação Finalizada,RN,PLUVIOMETRO3,06/01/2014,"0,0",null,2026-09-26T21:40:03.475Z,google_drive,LEITURA_PLUVIOMETROS.xlsx
190825,23/05/2025 10:53:42,Solicitação Finalizada,RN,PLUVIOMETRO4.,06/01/2014,"0,0",null,2026-09-26T21:40:03.475Z,google_drive,LEITURA_PLUVIOMETROS.xlsx
190825,23/05/2025 10:53:42,Solicitação Finalizada,RN,PLUVIOMETRO5,06/01/2014,"0,0",null,2026-09-26T21:40:03.475Z,google_drive,LEITURA_PLUVIOMETROS.xlsx
190826,23/05/2025 10:53:43,Solicitação Finalizada,RN,PLUVIOMETRO1,30/01/2014,"5,5",null,2026-09-26T21:40:03.475Z,google_drive,LEITURA_PLUVIOMETROS.xlsx
190826,23/05/2025 10:53:43,Solicitação Finalizada,RN,PLUVIOMETRO2,30/01/2014,"5,5",null,2026-09-26T21:40:03.475Z,google_drive,LEITURA_PLUVIOMETROS.xlsx
190826,23/05/2025 10:53:43,Solicitação Finalizada,RN,PLUVIOMETRO3,30/01/2014,"6,0",null,2026-09-26T21:40:03.475Z,google_drive,LEITURA_PLUVIOMETROS.xlsx
190826,23/05/2025 10:53:43,Solicitação Finalizada,RN,PLUVIOMETRO4.,30/01/2014,"3,0",null,2026-09-26T21:40:03.475Z,google_drive,LEITURA_PLUVIOMETROS.xlsx
190826,23/05/2025 10:53:43,Solicitação Finalizada,RN,PLUVIOMETRO5,30/01/2014,"1,5",null,2026-09-26T21:40:03.475Z,google_drive,LEITURA_PLUVIOMETROS.xlsx


Carga concluída. Prossiga para o notebook **MVP03-Bronze**.